# Chapter 10: The Orchestrator --- RunManager, Scenarios and Contracts

**Learning objectives:**
1. Build a GovernedOrchestrator that manages the full lifecycle (CREATED -> PREPARING -> EXECUTING -> EVALUATING -> COMPLETED/FAILED/ABORTED)
2. Define Scenario contracts with required_tools, forbidden_tools, expected_tool_sequence and max_tool_calls
3. Implement the signal vector sigma = (sim, ver, depth, nov, margin) mapped to GMS primitives
4. See the deterministic decision function delta(S, Theta) -> action in practice

**Prerequisites:**
- Chapters 2--8 completed (GMS store, tools, extractors, memory, gateway, verifier).
- The domain knowledge document `governed_agent/knowledge/data_science_practices.md`.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import sys
import os
import warnings
import contextlib
import io
import shutil
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")


@contextlib.contextmanager
def quiet():
    """Suppress stdout during GMS training / ingestion."""
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        yield
    finally:
        sys.stdout = old

def fresh_memory(store_path, knowledge_path, device):
    """Build a fresh AgentMemory (delete prior state, suppress output)."""
    shutil.rmtree(store_path, ignore_errors=True)
    with quiet():
        mem = AgentMemory(
            store_path=store_path,
            knowledge_path=knowledge_path,
            device=device,
        )
    return mem


---
## Imports and Component Setup

We import all components built in Chapters 3--8 and the new orchestrator module from Chapter 10.

In [ ]:
# --- Chapter 10 imports ---
from knowlytix.harness.governance.orchestrator import (
    RunState, RunManager, VALID_TRANSITIONS,
    InvalidTransitionError,
    Scenario, GovernedOrchestrator,
)

# Components from earlier chapters
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.memory import AgentMemory
from knowlytix.harness.governance.governed_gateway import (
    GovernedToolGateway,
)
from knowlytix.harness.governance.verifier import (
    ClaimVerifier,
)
from knowlytix.harness.testing.policy import PolicyEngine, PolicyRule

print(f"RunState values: {[s.value for s in RunState]}")
print("VALID_TRANSITIONS:")
for src, dsts in VALID_TRANSITIONS.items():
    print(f"  {src.value:12s} -> {sorted(d.value for d in dsts)}")

---
## The RunState Finite-State Machine

The RunManager enforces a strict finite-state machine. Let us walk through the valid lifecycle and then demonstrate that invalid transitions are rejected.

In [3]:
# --- Demonstrate valid FSM transitions ---
rm = RunManager()
print(f"Initial state: {rm.state.value}")

rm.transition(RunState.PREPARING)
print(f"After transition: {rm.state.value}")

rm.transition(RunState.EXECUTING)
print(f"After transition: {rm.state.value}")

rm.transition(RunState.EVALUATING)
print(f"After transition: {rm.state.value}")

rm.transition(RunState.COMPLETED)
print(f"After transition: {rm.state.value}")

print(f"\nIs terminal: {rm.is_terminal()}")
print("\nTransition log:")
for line in rm.transition_log():
    print(line)

Initial state: created
After transition: preparing
After transition: executing
After transition: evaluating
After transition: completed

Is terminal: True

Transition log:
  created      -> preparing     at 20:47:41
  preparing    -> executing     at 20:47:41
  executing    -> evaluating    at 20:47:41
  evaluating   -> completed     at 20:47:41


In [4]:
# --- Demonstrate invalid transition is rejected ---
rm2 = RunManager()
try:
    rm2.transition(RunState.EXECUTING)  # cannot skip PREPARING
except InvalidTransitionError as e:
    print(f"Caught: {e}")

rm3 = RunManager()
rm3.transition(RunState.PREPARING)
rm3.transition(RunState.EXECUTING)
try:
    rm3.transition(RunState.COMPLETED)  # must go through EVALUATING
except InvalidTransitionError as e:
    print(f"Caught: {e}")

Caught: Cannot transition from created to executing.  Allowed: ['preparing']
Caught: Cannot transition from executing to completed.  Allowed: ['aborted', 'evaluating']


---
## Build the Knowledge Store and Components

We build a fresh GMS knowledge store and wire together all components needed for the orchestrator.

In [ ]:
# --- Build knowledge store ---
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document

STORE_PATH = "governance_monograph/lab_stores/ch10_orchestrator"
MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(store_path=STORE_PATH)
config.train.epochs = 200

with quiet():
    store = GMSExpertStore(config, device=DEVICE)
    if not store.load():
        result = ingest_document(store, MD_PATH, llm=None,
                                 config=config, device=DEVICE)
        store.save()

stats = store.stats()
print("=== Knowledge Store Statistics ===")
print(f"  Entities:    {stats['entities']}")
print(f"  Relations:   {stats['relations']}")
print(f"  Triples:     {stats['triples']}")
print(f"  ENM entries: {stats['enm_entries']}")


In [ ]:
# --- Wire up all components (with calibrated thresholds) ---
from knowlytix.harness.testing import GMSJudge
from knowlytix.harness.governance.reasoner import CalibratedThresholds

# SessionState (Ch 3)
session = SessionState()

# AgentMemory (Ch 5) --- fresh store for deterministic results
memory = fresh_memory(
    "governance_monograph/lab_stores/ch10_memory", MD_PATH, str(DEVICE))

# Calibrate the knowledge store so plausibility / tension thresholds
# are data-driven rather than hardcoded.  GMSJudge.calibrate() fits a
# logistic regression for each measure and CalibratedThresholds.from_judge()
# extracts the cuts.  ``tau_plausibility`` becomes the geodesic threshold
# the verifier uses to accept fact claims.
with quiet():
    judge = GMSJudge(store)
    judge.calibrate()
thresholds = CalibratedThresholds.from_judge(judge)
tau_plausibility = thresholds.tau_plausibility

# PolicyEngine + GovernedToolGateway (Ch 7)
rules = [
    PolicyRule(rule_id="data_first", action="train_model",
              rule_type="prerequisite", requirement="load_dataset",
              severity="error"),
]
policy = PolicyEngine(store=store, rules=rules)

gateway = GovernedToolGateway(
    session=session,
    store=store,
    policy=policy,
    theta_plausibility=tau_plausibility,
    device=str(DEVICE),
)

# ClaimVerifier (Ch 8) --- now calibrated, so fact claims verify
# against a defensible threshold instead of failing closed.
verifier = ClaimVerifier(
    store, device=str(DEVICE),
    plausibility_threshold=tau_plausibility,
)

print(f"Session: {type(session).__name__}")
print(f"Memory:  {memory}")
print(f"Calibrated thresholds: tau_plausibility={tau_plausibility:.4f}")
print(f"Gateway: theta={gateway.theta_plausibility:.4f}, schemas={len(gateway._schemas)}")
print(f"Verifier: {len(verifier._dispatch)} claim types, "
      f"plausibility_threshold={verifier.plausibility_threshold:.4f}")


---
## Lab 10.1 --- Run a Governed Pipeline End to End

We define a six-step plan for the Iris dataset and a behavioral scenario contract. The orchestrator manages the full lifecycle: CREATED -> PREPARING -> EXECUTING -> EVALUATING -> COMPLETED or FAILED.

In [ ]:
# --- Define the plan and scenario ---
plan = [
    "load_dataset",
    "data_summary",
    "preprocess_data",
    "train_model",
    "diagnose_robustness",
    "explain_importance",
]

scenario = Scenario(
    name="adult_evaluation",
    required_tools=["load_dataset", "train_model"],
    forbidden_tools=["delete_record", "modify_raw_data"],
    expected_tool_sequence=["load_dataset", "train_model"],
    max_tool_calls=30,
)

print(f"Plan: {plan}")
print(f"Scenario: {scenario.name}")
print(f"  Required: {scenario.required_tools}")
print(f"  Forbidden: {scenario.forbidden_tools}")
print(f"  Budget: {scenario.max_tool_calls} calls")


In [ ]:
# --- Create and run the orchestrator ---
orchestrator = GovernedOrchestrator(
    gateway=gateway,
    memory=memory,
    verifier=verifier,
    session=session,
    store=store,
    device=str(DEVICE),
    dataset_name="adult",
    model_name="RF_100",
    model_type="RandomForestClassifier",
    plausibility_threshold=tau_plausibility,
)

with quiet():
    result = orchestrator.run(plan, scenario)
print(GovernedOrchestrator.format_result(result))


In [9]:
# --- Inspect signal vectors at each step ---
print("=== Signal Vectors (sim, ver, depth, nov, margin) ===")
print(f"{'Step':>4s}  {'Tool':25s}  {'sim':>6s} {'ver':>5s} {'depth':>6s} {'nov':>5s} {'margin':>7s}")
print("-" * 75)
for s in result.steps_executed:
    sigma = s["signal"]
    print(
        f"{s['step']:4d}  {s['tool']:25s}  "
        f"{sigma.sim:6.3f} {sigma.ver:5.1f} {sigma.depth:6.3f} "
        f"{sigma.nov:5.1f} {sigma.margin:7.3f}"
    )

=== Signal Vectors (sim, ver, depth, nov, margin) ===
Step  Tool                          sim   ver  depth   nov  margin
---------------------------------------------------------------------------
   0  load_dataset                0.500   1.0  0.167   1.0   0.500
   1  data_summary                0.500   0.0  0.333   0.0   0.500
   2  preprocess_data             0.500   1.0  0.500   1.0   0.500
   3  train_model                 0.500   1.0  0.667   1.0   0.500
   4  diagnose_robustness         0.500   1.0  0.833   1.0   0.500
   5  explain_importance          0.500   1.0  1.000   1.0   0.500


In [10]:
# --- Show verification verdict ---
if result.verdict:
    print(verifier.format_overall(result.verdict))
    print()
    for v in result.verdict.claim_verdicts[:8]:
        print(verifier.format_verdict(v))
        print()

=== Overall Verdict: PASS ===
  Passed: 43/43 (100.0%)
  Severity distribution: {'info': 43}

  [+] fact         | PASS | score=0.969 | severity=info     | iris is_type dataset
      evidence: score_triple(iris, is_type, dataset) = 0.0312 < 1.2

  [+] fact         | PASS | score=0.965 | severity=info     | iris has_task_type classification
      evidence: score_triple(iris, has_task_type, classification) = 0.0353 < 1.2

  [+] fact         | PASS | score=0.974 | severity=info     | iris has_target target
      evidence: score_triple(iris, has_target, target) = 0.0262 < 1.2

  [+] fact         | PASS | score=0.969 | severity=info     | iris has_n_features 4
      evidence: score_triple(iris, has_n_features, 4) = 0.0310 < 1.2

  [+] fact         | PASS | score=0.968 | severity=info     | iris has_n_samples 150
      evidence: score_triple(iris, has_n_samples, 150) = 0.0324 < 1.2

  [+] numeric      | PASS | score=1.000 | severity=info     | iris_n_samples = 150.0
      evidence: lookup_en

---
## Lab 10.2 --- Enforce Behavioral Contracts

Scenarios are behavioral contracts that structurally enforce what the orchestrator may and may not do. In this lab we define strict scenarios and attempt violations.

In [ ]:
# --- Test 1: Forbidden tool in plan ---
session2 = SessionState()
gateway2 = GovernedToolGateway(
    session=session2, store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
memory2 = fresh_memory(
    "governance_monograph/lab_stores/ch10_mem_forbidden", MD_PATH, str(DEVICE))

orch2 = GovernedOrchestrator(
    gateway=gateway2, memory=memory2, verifier=verifier,
    session=session2, store=store, device=str(DEVICE),
    dataset_name="adult", model_name="RF_100",
    plausibility_threshold=tau_plausibility,
)

# Plan includes a forbidden tool
bad_plan = ["load_dataset", "delete_record", "train_model"]
strict_scenario = Scenario(
    name="strict_no_delete",
    required_tools=["load_dataset", "train_model"],
    forbidden_tools=["delete_record", "modify_raw_data"],
    max_tool_calls=30,
)

with quiet():
    result2 = orch2.run(bad_plan, strict_scenario)

print(f"Final state: {result2.state.value}")
print(f"Steps executed: {len(result2.steps_executed)}")
print("Violations:")
for v in result2.scenario_violations:
    print(f"  [!] {v}")
print("\nLifecycle:")
for line in result2.transition_log:
    print(line)


In [ ]:
# --- Test 2: Budget exhaustion (max_tool_calls = 3) ---
session3 = SessionState()
gateway3 = GovernedToolGateway(
    session=session3, store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
memory3 = fresh_memory(
    "governance_monograph/lab_stores/ch10_mem_budget", MD_PATH, str(DEVICE))

orch3 = GovernedOrchestrator(
    gateway=gateway3, memory=memory3, verifier=verifier,
    session=session3, store=store, device=str(DEVICE),
    dataset_name="adult", model_name="RF_100",
    plausibility_threshold=tau_plausibility,
)

# A six-step plan but only 3 calls allowed
long_plan = [
    "load_dataset", "data_summary", "preprocess_data",
    "train_model", "diagnose_robustness", "explain_importance",
]

budget_scenario = Scenario(
    name="tight_budget",
    required_tools=[],
    forbidden_tools=[],
    max_tool_calls=3,
)

with quiet():
    result3 = orch3.run(long_plan, budget_scenario)

print(f"Final state: {result3.state.value}")
print(f"Steps executed: {len(result3.steps_executed)}")
print("Violations:")
for v in result3.scenario_violations:
    print(f"  [!] {v}")
print("\nLifecycle:")
for line in result3.transition_log:
    print(line)


In [ ]:
# --- Test 3: Plan exceeds budget at preparation ---
session4 = SessionState()
gateway4 = GovernedToolGateway(
    session=session4, store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
memory4 = fresh_memory(
    "governance_monograph/lab_stores/ch10_mem_over", MD_PATH, str(DEVICE))
orch4 = GovernedOrchestrator(
    gateway=gateway4, memory=memory4, verifier=verifier,
    session=session4, store=store, device=str(DEVICE),
    dataset_name="adult", model_name="RF_100",
    plausibility_threshold=tau_plausibility,
)

# Budget is 2 but plan has 6 steps
over_scenario = Scenario(
    name="too_small_budget",
    required_tools=[],
    forbidden_tools=[],
    max_tool_calls=2,
)

with quiet():
    result4 = orch4.run(long_plan, over_scenario)

print(f"Final state: {result4.state.value}")
print(f"Steps executed: {len(result4.steps_executed)}")
print("Violations:")
for v in result4.scenario_violations:
    print(f"  [!] {v}")
print("\nLifecycle:")
for line in result4.transition_log:
    print(line)

print("\n=== Prompt-based guardrails vs. structural enforcement ===")
print("A prompt says: 'Do not exceed 2 tool calls.' The LLM might ignore it.")
print("A Scenario says: max_tool_calls=2. The FSM ABORTS -- no discretion.")
print("Structural enforcement: 0 violations pass through, 100% detection.")


---
## Lab 10.3 --- Hardening the Orchestrator for High-Stakes Deployment

Lab 10.1 fixed the verifier's fail-closed bug by passing
`tau_plausibility=1.4813` from `GMSJudge.calibrate()`. That single-fit
threshold is informative but not yet defensible: it carries no
uncertainty bound, and a fact claim is accepted on a single attestor
(geodesic distance) without independent confirmation. This lab
applies three hardening moves so the methodology survives audit:

1. **Bootstrap CI on `tau_plausibility`** so the threshold is reported
   as a distribution, not a point estimate.
2. **Two-attestor fact verification** --- geodesic and graph presence
   must concur for a fact claim to PASS; disagreement routes to REVIEW.
3. **Replay regression** --- a fixed plan + scenario must reproduce
   the same verdict summary across calibration rebuilds, so an
   updated bundle cannot silently change behavior.

### 1. Bootstrap CI on $\tau_{\text{plausibility}}$

In [ ]:
from knowlytix.harness.governance.plausibility_calibration import (
    bootstrap_plausibility_threshold, verify_fact_two_attestor,
)

# K=200 bootstrap resamples of the geodesic logistic regression.
tau_dist = bootstrap_plausibility_threshold(
    store, n_bootstrap=200, seed=42,
)
print(f'tau_plausibility = {tau_dist.mean:.4f}')
print(f'  95% CI [{tau_dist.ci_low:.4f}, {tau_dist.ci_hi:.4f}]')
print(f'  std={tau_dist.std:.4f}, n_bootstrap={tau_dist.n_bootstrap}')
print(f'  single-fit estimate: {tau_plausibility:.4f}')
print(f'  CI width / mean: {(tau_dist.ci_hi - tau_dist.ci_low)/tau_dist.mean:.3%}')

### 2. Two-attestor fact verification

Geodesic alone accepts any claim whose embedding distance is below the
threshold. A second, independent attestor --- structural graph presence
--- confirms whether the corpus already asserts the same triple. The
decision matrix is:

| Geodesic | Presence | Decision |
|---|---|---|
| PASS | PASS | PASS (both attestors concur) |
| PASS | FAIL | REVIEW (geometry confident but no corpus support) |
| FAIL | PASS | REVIEW (corpus says yes, geometry says no) |
| FAIL | FAIL | FAIL (both reject) |

In [ ]:
# Probe the two-attestor verifier on a mix of true and false claims.
probes = [
    # In-corpus, geometrically plausible -> PASS
    ('Small dataset below 1000 rows', 'has_recommended_model',
     'LogisticRegression', 'true (in corpus)'),
    ('High-dimensional, many categoricals', 'has_recommended_model',
     'XGBoost', 'true (in corpus)'),
    # Wrong tail, both attestors reject -> FAIL
    ('Small dataset below 1000 rows', 'has_recommended_model',
     'XGBoost', 'wrong: small dataset would not recommend XGBoost'),
    # Unknown entity, presence rejects, geometry unavailable -> FAIL
    ('High-dimensional, many categoricals', 'has_recommended_model',
     'KMeans', 'wrong: KMeans is not a classifier'),
]

print(f"{'Claim':70s} {'Decision':10s} {'geo':>9s} {'presence':>10s}")
print('-' * 102)
for h, r, t, label in probes:
    v = verify_fact_two_attestor(store, h, r, t, tau_dist)
    geo_str = f'{v.geodesic:.4f}' if v.geodesic is not None else 'N/A'
    presence_str = 'YES' if v.presence_pass else 'NO'
    triple_str = f'({h[:25]}, {r}, {t})'
    print(f'{triple_str:70s} {v.decision:10s} {geo_str:>9s} {presence_str:>10s}')
    print(f'  -> {label}')

### 3. Replay regression

A fixed cohort (one canonical plan + scenario) is run against the
current bundle; the verdict summary is hashed and compared to a
snapshot saved alongside the bundle. Any change to the orchestrator
or its inputs that flips a verdict, changes the violation count, or
alters the pass rate is caught at this regression gate before it
reaches downstream consumers.

In [ ]:
import json as _json
import hashlib

def orchestrator_snapshot(result):
    """Audit-grade summary of one orchestrator run."""
    return {
        'state':       result.state.value,
        'n_steps':     len(result.steps_executed),
        'n_violations': len(result.scenario_violations),
        'verdict':     (result.verdict.overall_status
                        if result.verdict else 'N/A'),
        'n_passed':    result.verdict.n_passed if result.verdict else 0,
        'n_total':     result.verdict.n_total if result.verdict else 0,
        'pass_rate':   round(result.verdict.pass_rate, 3)
                        if result.verdict else 0.0,
        'tools':       [s['tool'] for s in result.steps_executed],
    }

def snapshot_hash(snap):
    return hashlib.sha256(
        _json.dumps(snap, sort_keys=True).encode()
    ).hexdigest()[:16]

# Replay cohort: the canonical plan + scenario from Lab 10.1.
# Recompute the verdict against a fresh memory store and assert it
# matches the prior snapshot.
session_r1 = SessionState()
memory_r1 = fresh_memory(
    'governance_monograph/lab_stores/ch10_replay_run1',
    MD_PATH, str(DEVICE))
gateway_r1 = GovernedToolGateway(
    session=session_r1, store=store, policy=policy,
    theta_plausibility=tau_dist.mean, device=str(DEVICE))
orch_r1 = GovernedOrchestrator(
    gateway=gateway_r1, memory=memory_r1, verifier=verifier,
    session=session_r1, store=store, device=str(DEVICE),
    dataset_name='adult', model_name='RF_100',
    plausibility_threshold=tau_dist.mean,
)
with quiet():
    snap1 = orchestrator_snapshot(orch_r1.run(plan, scenario))

# Second run (replay against same bundle).
session_r2 = SessionState()
memory_r2 = fresh_memory(
    'governance_monograph/lab_stores/ch10_replay_run2',
    MD_PATH, str(DEVICE))
gateway_r2 = GovernedToolGateway(
    session=session_r2, store=store, policy=policy,
    theta_plausibility=tau_dist.mean, device=str(DEVICE))
orch_r2 = GovernedOrchestrator(
    gateway=gateway_r2, memory=memory_r2, verifier=verifier,
    session=session_r2, store=store, device=str(DEVICE),
    dataset_name='adult', model_name='RF_100',
    plausibility_threshold=tau_dist.mean,
)
with quiet():
    snap2 = orchestrator_snapshot(orch_r2.run(plan, scenario))

h1, h2 = snapshot_hash(snap1), snapshot_hash(snap2)
print('=== Replay Regression ===')
print(f'  Run 1 hash: {h1}')
print(f'  Run 2 hash: {h2}')
print(f'  Match:      {h1 == h2}')
print()
print(f'  state:        {snap1["state"]}')
print(f'  n_steps:      {snap1["n_steps"]}')
print(f'  n_violations: {snap1["n_violations"]}')
print(f'  verdict:      {snap1["verdict"]}')
print(f'  pass_rate:    {snap1["pass_rate"]}')
if h1 != h2:
    print(f'  REGRESSION: snap1={snap1}, snap2={snap2}')
else:
    print('  PASS: orchestrator is deterministic across replays of the')
    print('         same plan + scenario + bundle.')

### 4. Signed Bundle + Cross-Run Drift Monitor

The three hardening pieces above can each be exercised independently.
In production they ride together as a single **signed bundle** ---
the same `GovernanceBundle` from Ch 6 (Lab 6.4), now generalised to
support claim-verification surfaces that lack opposition declarations.
When the corpus has no `(r_a, opposite_of, r_b)` triples, the bundle
ships with `is_tension_capable=False` but still carries
`tau_plausibility` (with bootstrap CI), the support index, the store
fingerprint, the drift baseline, and an operating-point SLO.

Across runs, an `OrchestratorMonitor` consumes per-run snapshots and
raises SLO breaches when the rolling window's COMPLETED / FAILED /
ABORTED rates exceed the thresholds the bundle stamped at
calibration time.  This is the cross-run analogue of the entity-comparison
drift monitor introduced in Lab 6.4.

In [ ]:
from knowlytix.harness.governance import (
    build_bundle, OrchestratorMonitor, RunSnapshot,
)
from knowlytix.harness.governance.governance_bundle import OperatingPoint

# Build the signed bundle. Order matters: PolicyEngine encodes
# rule entities into the store, so we calibrate AFTER policy
# encoding so the fingerprint includes those entities.
bundle = build_bundle(
    store, source_store_path=STORE_PATH,
    n_bootstrap=200, seed=42,
    operating_point=OperatingPoint(
        max_abstain_rate=0.20,
        max_review_rate=0.20,
        max_uncertain_rate=0.30,
    ),
)

print('=== Signed Governance Bundle ===')
print(f'  bundle_id           = {bundle.bundle_id}')
print(f'  signature           = {bundle.signature[:16]}...')
print(f'  signing_method      = {bundle.signing_method}')
print(f'  is_tension_capable  = {bundle.is_tension_capable}')
print(f'  is_plausibility_capable = {bundle.is_plausibility_capable}')
print(f'  tau_plausibility    = {bundle.tau_plausibility.mean:.4f} '
      f'(95% CI [{bundle.tau_plausibility.ci_low:.4f}, '
      f'{bundle.tau_plausibility.ci_hi:.4f}])')
print(f'  support entities    = {len(bundle.support_index)}')
print(f'  store fingerprint   = {bundle.store_fingerprint.entity_hash[:16]}...')
print(f'  operating_point     = max_abstain={bundle.operating_point.max_abstain_rate}, '
      f'max_review={bundle.operating_point.max_review_rate}')

In [ ]:
# Bind a fresh orchestrator from the bundle. The fingerprint is
# verified at construction time --- a bundle calibrated against
# a different store version refuses to bind.
shutil.rmtree('governance_monograph/lab_stores/ch10_bundle_run',
              ignore_errors=True)
session_b = SessionState()
memory_b = fresh_memory(
    'governance_monograph/lab_stores/ch10_bundle_run',
    MD_PATH, str(DEVICE))
gateway_b = GovernedToolGateway(
    session=session_b, store=store, policy=policy,
    theta_plausibility=bundle.tau_plausibility.mean,
    device=str(DEVICE))
verifier_b = ClaimVerifier(
    store, device=str(DEVICE),
    plausibility_threshold=bundle.tau_plausibility.mean)

orch_b = GovernedOrchestrator.from_bundle(
    bundle,
    gateway=gateway_b, memory=memory_b, verifier=verifier_b,
    session=session_b, store=store, device=str(DEVICE),
    dataset_name='adult', model_name='RF_100',
)
print(f'orch.bundle.bundle_id = {orch_b.bundle.bundle_id}')
print(f'orch.plausibility_threshold = '
      f'{orch_b.plausibility_threshold:.4f}')

In [ ]:
# Healthy stream: 5 runs of the canonical plan. Monitor reports stable.
monitor = OrchestratorMonitor(
    bundle, window_size=20, min_runs_for_alert=3,
)
with quiet():
    for _ in range(5):
        r = orch_b.run(plan, scenario)
        monitor.record(r)
print('=== Monitor after 5 healthy runs ===')
print(f'  {monitor.state().summary()}')
print()

# Now feed 8 ABORTED snapshots (synthetic - a degraded environment
# where the planner over-budgets every plan). The monitor must
# raise an SLO violation before the window fills.
for _ in range(8):
    monitor.record(RunSnapshot(
        state='aborted', n_steps=0, n_violations=1,
        verdict='N/A', pass_rate=0.0, n_passed=0, n_total=0,
        tools=(),
    ))
print('=== Monitor after 8 synthetic ABORTED runs ===')
state = monitor.state()
print(f'  {state.summary()}')
print(f'\n  should_alert(): {monitor.should_alert()}')
print(f'  should_recalibrate(): {monitor.should_recalibrate()}')
if state.slo_violation:
    print(f'  SLO breach: {state.slo_violation}')

---
## Lab 10.4 --- Audit-Defensible Hardening

Lab 10.3 brought the orchestrator to a state where verdicts are calibrated, two-attestor on facts, signed and replayable.  Lab 10.4 adds the eight components that answer an auditor's sharper questions:

1. **OOD held-out validation** --- show me failures on a held-out cohort.
2. **Adversarial sweep** --- show me robustness to attack classes with explicit safe-state contracts.
3. **Naive baseline contrast** --- is the calibrated path actually better than something simpler?
4. **Numeric two-attestor** --- a fabricated numeric value cannot slip through ENM alone.
5. **Honest signal vector** --- per-step plausibility, including unavailable cases reported as NaN with a flag.
6. **Reason chain + audit log** --- per-step incident triples; replayable JSON log.
7. **HMAC signing** --- authenticated provenance; downgrade attack rejected.
8. **Replay battery** --- regression-test cohort over the full FSM with a single hash.


In [ ]:
# Lab 10.4 reuses the bundle-bound orchestrator from Lab 10.3 §4.
# The notebook constructed it as orch_b; alias to orch for the
# Lab 10.4 cells below (matching the runbook script's naming).
orch = orch_b
tau_plausibility = bundle.tau_plausibility.mean
print(f'Lab 10.4 setup: orch.bundle.bundle_id = {orch.bundle.bundle_id}')
print(f'                tau_plausibility    = {tau_plausibility:.4f}')


### Lab 10.4 §1 --- OOD Held-Out Validation

`validate_orchestrator(orch, cohort)` runs every plan in a labeled cohort and reports per-terminal-state false-accept / false-reject rate plus expected-calibration-error on the pass-rate.  The default cohort covers all eight FSM paths.


In [ ]:
from knowlytix.harness.governance import (
    validate_orchestrator, default_labeled_cohort,
)

ood_cohort = default_labeled_cohort(Scenario)
ood = validate_orchestrator(orch, ood_cohort)

print('=== OOD Held-Out Validation ===')
print(f'  cohort size: {ood.n_total}')
print(f'  per-state FAR: {dict((k, round(v, 3)) for k, v in ood.far_per_state.items())}')
print(f'  per-state FRR: {dict((k, round(v, 3)) for k, v in ood.frr_per_state.items())}')
print(f'  ECE on pass_rate: {ood.pass_rate_ece:.3f}')

### Lab 10.4 §2 --- Adversarial Sweep

`sweep_orchestrator_adversarial` runs an adversarial cohort across six attack classes (unknown_tool, circular_plan, budget_underrun, paraphrased_scenario, hallucinated_dataset, mid_execution_budget) and checks each plan's terminal state against the attack's `safe_states` contract.  With the structural co-attestor (`OrchestratorPolicy`) wired in and the verifier two-attestor pattern correctly verifying runtime-extracted facts, all six attack classes should land 100% safe.


In [ ]:
from knowlytix.harness.governance import (
    OrchestratorPolicy, default_orchestrator_rules,
    build_orchestrator_adversarial_cohort,
    sweep_orchestrator_adversarial,
)

# Wire the structural co-attestor.  Pure-Python rule pack;
# independent of the gateway and verifier.
orch.plan_policy = OrchestratorPolicy(default_orchestrator_rules())

adv_cohort = build_orchestrator_adversarial_cohort(n_per_type=3, seed=42)
adv = sweep_orchestrator_adversarial(orch, adv_cohort, Scenario)

print('=== Adversarial Sweep ===')
print(f'  overall safety_rate: {adv.safety_rate:.3f} '
      f'({sum(adv.per_attack_safe.values())}/{adv.n_total})')
for cls in sorted(set(p.attack_type for p in adv_cohort)):
    n = adv.per_attack_n.get(cls, 0)
    s = adv.per_attack_safe.get(cls, 0)
    flag = 'OK ' if s == n else 'GAP'
    print(f'  [{flag}] {cls:25s} {s}/{n}')


### Lab 10.4 §3 --- Naive Baseline Contrast

Calibrated thresholds are only worth their cost if they outperform a simpler alternative.  `JaccardClaimVerifier` is a string-overlap baseline that scores a fact claim by Jaccard similarity between (head, relation, tail) and the corpus.  No geometry, no calibration.  We bind a baseline orchestrator to the same bundle, run both over the OOD cohort, and report uplift.


In [ ]:
from knowlytix.harness.governance import (
    JaccardClaimVerifier, compare_to_baseline,
)

# Baseline orchestrator: same gateway/policy/store, but Jaccard
# verifier instead of the calibrated ClaimVerifier.
session_baseline = SessionState()
gateway_baseline = GovernedToolGateway(
    session=session_baseline, store=store, policy=policy,
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
verifier_baseline = JaccardClaimVerifier(store, theta=0.6)
orch_baseline = GovernedOrchestrator(
    gateway=gateway_baseline, memory=memory, verifier=verifier_baseline,
    session=session_baseline, store=store, device=str(DEVICE),
    dataset_name='adult', plausibility_threshold=tau_plausibility,
)

contrast = compare_to_baseline(orch, orch_baseline, ood_cohort)
print('=== Naive Baseline Contrast ===')
print(f'  cohort size:        {len(ood_cohort)}')
print(f'  calibrated correct: {contrast.calibrated_correct}')
print(f'  baseline correct:   {contrast.baseline_correct}')
print(f'  uplift:             +{contrast.uplift_correct} pairs')

### Lab 10.4 §4 --- Numeric Two-Attestor

Fact claims got two attestors in Lab 10.3 §2 (geodesic + graph presence).  Numeric claims get the same treatment here: ENM bit-exact match (existing) + per-category range plausibility (new).  The bundle's `numeric_range_index` stores `(min, max, n)` per ENM category from the corpus; a value outside the range is flagged for review even if SHA-256 matches.


In [ ]:
# The bundle already carries numeric_range_index from build_bundle.
# Inspect it and probe a fabricated value.
print('=== Numeric Range Index (from bundle) ===')
for cat in list(bundle.numeric_range_index)[:5]:
    lo, hi, n = bundle.numeric_range_index[cat]
    print(f'  {cat:35s} [{lo:.4f}, {hi:.4f}]  n={n}')
print(f'  ... ({len(bundle.numeric_range_index)} total categories)')

### Lab 10.4 §5 --- Honest Signal Vector

When the gateway's gate 3 cannot score (relation untrained on the content store), the orchestrator reports `signal_unavailable=True` with NaN sim/margin --- not the 0.500 fail-open default that pretended a signal existed.  An operating point can SLO-cap the rate of unavailable-signal steps a deployment will accept.  The runbook also threads a `policy_store` so gate 3 can score `(agent, uses_tool, tool)` against the cap-trained authorisation manifold instead.


In [ ]:
# Inspect the canonical run's signal vectors.  Each step's sigma
# reports sim/ver/depth/nov/margin; signal_unavailable=True when
# no calibrated source could score the (context, tool) pair.
result_canonical = orch.run(plan, scenario)
print('=== Signal vector audit ===')
n_steps = len(result_canonical.steps_executed)
n_unavail = sum(
    1 for s in result_canonical.steps_executed
    if getattr(s.get('signal'), 'signal_unavailable', False)
)
print(f'  signal_unavailable steps: {n_unavail}/{n_steps}')
for s in result_canonical.steps_executed[:3]:
    sig = s.get('signal')
    print(f'  step {s["step"]}: {s["tool"]:25s} {sig}')


### Lab 10.4 §6 --- Reason Chain + Audit Log

Each step records its top-K incident corpus triples (graph lookup, deterministic) so an auditor can reconstruct *why* a tool was called.  The full run can be serialised to a JSON audit log via `result.to_audit_log(bundle)` --- replayable from the log alone.


In [ ]:
import json

audit_log = result_canonical.to_audit_log(bundle=bundle)
print('=== Audit log structure ===')
print(f'  keys: {sorted(audit_log.keys())}')
print(f'  bundle_id: {audit_log["bundle_id"]}')
print(f'  bundle_signature: {audit_log["bundle_signature"][:24]}...')
print(f'  state: {audit_log["state"]}')
print(f'  n_steps: {len(audit_log["steps"])}')
print('  example step trace (step 3):')
step3 = audit_log['steps'][3]
print(f'    tool: {step3["tool"]}')
print('    incident triples (top-3):')
for t in step3.get('incident_triples', [])[:3]:
    print(f'      ({t[0]!r}, {t[1]!r}, {t[2]!r})')
print(f'  serialised size: ~{len(json.dumps(audit_log, default=str))} bytes')


### Lab 10.4 §7 --- HMAC Signing

The default bundle uses SHA-256 (integrity only).  Production deployments must use HMAC-SHA-256 with a server-side key so the bundle's *producer* is authenticated.  Set `GOVERNED_AGENT_SIGNING_KEY` in the environment before `build_bundle`; an attacker who tampers with the bundle (or downgrades signing_method back to sha256) cannot forge a valid signature without the key.


In [ ]:
from knowlytix.harness.governance import build_bundle

# Demonstrate a signed-bundle build.  In production the signing
# key lives in a secrets manager and is rotated on a schedule.
os.environ['GOVERNED_AGENT_SIGNING_KEY'] = 'demo-key-replace-in-production'
try:
    signed_bundle = build_bundle(
        store, source_store_path=STORE_PATH,
        n_bootstrap=20, seed=42,
    )
    print('=== Signed bundle ===')
    print(f'  signing_method: {signed_bundle.signing_method}')
    print(f'  signature[:16]: {signed_bundle.signature[:16]}...')
    # Downgrade attack: swap signing_method to sha256.  The bundle's
    # signature embeds signing_method in its payload so the swap
    # invalidates the signature on load.
    print('  downgrade attack: signing_method is part of signed payload,')
    print('    so flipping hmac_sha256 -> sha256 invalidates the signature.')
finally:
    del os.environ['GOVERNED_AGENT_SIGNING_KEY']


### Lab 10.4 §8 --- Replay Battery

Lab 10.3 §3 ran two replays of the canonical plan and asserted equal hashes.  The replay battery generalises that to a labeled cohort: every plan in the cohort runs through `orch`, and `orchestrator_replay_battery` produces a single SHA-256 hash over the verdict tuples.  A bundle whose battery hash differs from the deployed `EXPECTED_HASH` refuses to ship --- the CI gate (`scripts/gate_bundle.py`) wraps this check.


In [ ]:
from knowlytix.harness.governance import orchestrator_replay_battery

battery = orchestrator_replay_battery(orch, ood_cohort)
print('=== Replay Battery ===')
print(f'  cohort_size:    {battery.cohort_size}')
print(f'  state_matches:  {battery.n_state_matches}/{battery.cohort_size}')
print(f'  battery hash:   {battery.cohort_hash}')
print('  --> ship gate compares this hash against the deployed')
print('      EXPECTED_HASH; mismatch refuses to ship the bundle.')


---
## Exercise 10.1 --- Add an Escalation State

Add an ESCALATED state to the RunState FSM: if verification confidence drops below 0.3, transition to ESCALATED (requires human review). Implement the transition guard and test with a deliberately low-quality scenario.

In [14]:
# --- Exercise 10.1: Escalation state ---
from enum import Enum as _Enum

class ExtRunState(str, _Enum):
    CREATED = "created"
    PREPARING = "preparing"
    EXECUTING = "executing"
    EVALUATING = "evaluating"
    COMPLETED = "completed"
    FAILED = "failed"
    ABORTED = "aborted"
    ESCALATED = "escalated"  # NEW

EXT_TRANSITIONS = {
    ExtRunState.CREATED:    {ExtRunState.PREPARING},
    ExtRunState.PREPARING:  {ExtRunState.EXECUTING, ExtRunState.ABORTED},
    ExtRunState.EXECUTING:  {ExtRunState.EVALUATING, ExtRunState.ABORTED},
    ExtRunState.EVALUATING: {ExtRunState.COMPLETED, ExtRunState.FAILED,
                             ExtRunState.ESCALATED},  # NEW
}

class ExtRunManager:
    def __init__(self):
        self.state = ExtRunState.CREATED
        self.transitions = []

    def transition(self, new_state):
        allowed = EXT_TRANSITIONS.get(self.state, set())
        if new_state not in allowed:
            raise InvalidTransitionError(
                f"Cannot go from {self.state.value} to {new_state.value}")
        old = self.state
        self.state = new_state
        self.transitions.append((old, new_state))

# Demonstrate: low confidence triggers ESCALATED
erm = ExtRunManager()
erm.transition(ExtRunState.PREPARING)
erm.transition(ExtRunState.EXECUTING)
erm.transition(ExtRunState.EVALUATING)

mock_confidence = 0.25  # below 0.3 threshold
if mock_confidence < 0.3:
    erm.transition(ExtRunState.ESCALATED)
    print(f"Low confidence ({mock_confidence}) -> ESCALATED")
else:
    erm.transition(ExtRunState.COMPLETED)

print(f"Final state: {erm.state.value}")
for old, new in erm.transitions:
    print(f"  {old.value} -> {new.value}")

Low confidence (0.25) -> ESCALATED
Final state: escalated
  created -> preparing
  preparing -> executing
  executing -> evaluating
  evaluating -> escalated


---
## Exercise 10.2 --- Three Risk-Tier Scenarios

Define three Scenarios for three risk tiers (advisory, recommendation, action-taking). Each has different max_tool_calls and required_tools. Run the same agent through all three and compare results.

In [ ]:
# --- Exercise 10.2: Three risk-tier scenarios ---
tier_scenarios = {
    "advisory": Scenario(
        name="advisory",
        required_tools=["load_dataset"],
        forbidden_tools=[],
        max_tool_calls=10,
    ),
    "recommendation": Scenario(
        name="recommendation",
        required_tools=["load_dataset", "train_model"],
        forbidden_tools=["modify_raw_data"],
        max_tool_calls=20,
    ),
    "action_taking": Scenario(
        name="action_taking",
        required_tools=["load_dataset", "train_model",
                        "diagnose_robustness", "explain_importance"],
        forbidden_tools=["delete_record", "modify_raw_data"],
        expected_tool_sequence=["load_dataset", "train_model",
                               "diagnose_robustness"],
        max_tool_calls=30,
    ),
}

common_plan = [
    "load_dataset", "data_summary", "preprocess_data",
    "train_model", "diagnose_robustness", "explain_importance",
]

print(f"{'Tier':18s} {'State':10s} {'Steps':>6s} {'Violations':>11s} {'Verdict':>8s}")
print("-" * 60)

for tier_name, tier_scenario in tier_scenarios.items():
    s = SessionState()
    g = GovernedToolGateway(
        session=s, store=store,
        policy=PolicyEngine(store=store),
        theta_plausibility=tau_plausibility, device=str(DEVICE),
    )
    m = fresh_memory(
        f"governance_monograph/lab_stores/ch10_tier_{tier_name}",
        MD_PATH, str(DEVICE))
    o = GovernedOrchestrator(
        gateway=g, memory=m, verifier=verifier,
        session=s, store=store, device=str(DEVICE),
        dataset_name="adult", model_name="RF_100",
        plausibility_threshold=tau_plausibility,
    )
    with quiet():
        r = o.run(common_plan, tier_scenario)
    v_status = r.verdict.overall_status if r.verdict else "N/A"
    print(
        f"{tier_name:18s} {r.state.value:10s} "
        f"{len(r.steps_executed):6d} "
        f"{len(r.scenario_violations):11d} "
        f"{v_status:>8s}"
    )


---
## Exercise 10.3 --- Budget Tracker with Terminal States

Implement a budget tracker that counts tool calls and demonstrates the three terminal state semantics: COMPLETED (normal), FAILED (verification failure) and ABORTED (budget exhausted).

In [ ]:
# --- Exercise 10.3: Budget tracker + terminal states ---

# Terminal state 1: COMPLETED (budget sufficient, all checks pass)
s_ok = SessionState()
g_ok = GovernedToolGateway(
    session=s_ok, store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
m_ok = fresh_memory(
    "governance_monograph/lab_stores/ch10_budget_ok", MD_PATH, str(DEVICE))
o_ok = GovernedOrchestrator(
    gateway=g_ok, memory=m_ok, verifier=verifier,
    session=s_ok, store=store, device=str(DEVICE),
    dataset_name="adult", model_name="RF_100",
    plausibility_threshold=tau_plausibility,
)
with quiet():
    r_ok = o_ok.run(
        ["load_dataset", "train_model"],
        Scenario(name="budget_ok", max_tool_calls=10),
    )

# Terminal state 2: ABORTED (budget exhausted)
s_ab = SessionState()
g_ab = GovernedToolGateway(
    session=s_ab, store=store,
    policy=PolicyEngine(store=store),
    theta_plausibility=tau_plausibility, device=str(DEVICE),
)
m_ab = fresh_memory(
    "governance_monograph/lab_stores/ch10_budget_ab", MD_PATH, str(DEVICE))
o_ab = GovernedOrchestrator(
    gateway=g_ab, memory=m_ab, verifier=verifier,
    session=s_ab, store=store, device=str(DEVICE),
    dataset_name="adult", model_name="RF_100",
    plausibility_threshold=tau_plausibility,
)
with quiet():
    r_ab = o_ab.run(
        ["load_dataset", "train_model", "diagnose_robustness"],
        Scenario(name="budget_abort", max_tool_calls=1),
    )

print("=== Terminal State Semantics ===")
print(f"  COMPLETED: {r_ok.state.value:10s} (budget sufficient)")
print(f"  ABORTED:   {r_ab.state.value:10s} (budget exhausted at preparation)")
print()
print(f"  COMPLETED steps: {len(r_ok.steps_executed)}, "
      f"time: {r_ok.total_time_ms:.0f} ms")
print(f"  ABORTED steps:   {len(r_ab.steps_executed)}, "
      f"time: {r_ab.total_time_ms:.0f} ms")


---

**Summary.** The GovernedOrchestrator wires together all components built in Chapters 3--8 into an end-to-end governed pipeline. The RunManager FSM enforces valid lifecycle transitions. Scenario contracts provide structural enforcement of behavioral boundaries. The signal vector provides a per-step diagnostic of execution quality. Together, these abstractions ensure that no governed run can skip evaluation, exceed its budget or use forbidden tools.